# DWBI Week 5 — The ETL Process

**Ali Raza** · `su92-bsdsm-f23-018` · **BSDS-7A** · Superior University

PakFreight's Week-2 merge rebuilt as a pipeline that reruns: 37 rows land, 35 append, 34 load — 578,650 kg and PKR 4,808,000, exactly. Every number below is computed from `data/source/`, `data/dims/` or `data/night2/` in the notebook itself; nothing is typed in as a constant except the assertion target it is checked against.

## T5 — The locale trap (12 marks)

The marine file writes its dates as text in `dd/MM/yyyy`. On a machine set to
United States regional settings, Power Query reads `01/02/2026` as the **second
of January** — and pandas' naive `pd.to_datetime` makes the exact same mistake,
because it guesses month-first. This is the only defect of the ten that produces
a plausible, confidently wrong answer with no error message anywhere.

In [1]:
# T5: read the twelve marine date strings both ways, exactly as the two locales would
import pandas as pd
from datetime import datetime
marine = pd.read_csv("data/source/marine_shipments.csv", dtype=str, keep_default_na=False)
strings = list(marine.Ship_Date)
def read_gb(s):
    return datetime.strptime(s, "%d/%m/%Y").strftime("%Y-%m-%d")
def read_us(s):
    try:
        return datetime.strptime(s, "%m/%d/%Y").strftime("%Y-%m-%d")
    except ValueError:
        return "Error"
rows = []
for s in strings:
    gb, us = read_gb(s), read_us(s)
    if us == "Error":
        verdict, us_month = "Error", "Error"
    elif us == gb:
        verdict, us_month = "Right by luck", datetime.strptime(us, "%Y-%m-%d").strftime("%B")
    else:
        verdict, us_month = "Wrong month", datetime.strptime(us, "%Y-%m-%d").strftime("%B")
    rows.append((s, gb, us, verdict, us_month))
t5 = pd.DataFrame(rows, columns=["raw", "en-GB", "en-US", "verdict", "us_month"])
print(t5.to_string(index=False))
print("\nverdicts:", t5.verdict.value_counts().to_dict())
print("readable en-US rows:", (t5.verdict != "Error").sum(),
      "| distinct en-US months:", t5.loc[t5.verdict != "Error", "us_month"].nunique(),
      "| August present:", "August" in set(t5.us_month))
assert t5.verdict.value_counts().to_dict() == {"Wrong month": 10, "Right by luck": 1, "Error": 1}
assert t5.loc[t5.verdict != "Error", "us_month"].nunique() == 11
assert "August" not in set(t5.us_month)

       raw      en-GB      en-US       verdict  us_month
01/02/2026 2026-02-01 2026-01-02   Wrong month   January
02/02/2026 2026-02-02 2026-02-02 Right by luck  February
03/02/2026 2026-02-03 2026-03-02   Wrong month     March
04/02/2026 2026-02-04 2026-04-02   Wrong month     April
05/02/2026 2026-02-05 2026-05-02   Wrong month       May
06/02/2026 2026-02-06 2026-06-02   Wrong month      June
07/02/2026 2026-02-07 2026-07-02   Wrong month      July
09/02/2026 2026-02-09 2026-09-02   Wrong month September
10/02/2026 2026-02-10 2026-10-02   Wrong month   October
11/02/2026 2026-02-11 2026-11-02   Wrong month  November
12/02/2026 2026-02-12 2026-12-02   Wrong month  December
13/02/2026 2026-02-13      Error         Error     Error

verdicts: {'Wrong month': 10, 'Right by luck': 1, 'Error': 1}
readable en-US rows: 11 | distinct en-US months: 11 | August present: False


**Analysis.** The computation confirms the workbook self-check exactly: 10 wrong months, 1 right by luck (02/02, where day and month are the same number), and 1 error (13/02, because there is no month 13). The eleven readable US rows land in eleven different months — so a report grouped by month would show a plausible-looking spread across the year for what was, in reality, a single February's shipping.

![Month distribution](charts/t5_locale.png)

| # | Raw string | en-GB · dd/MM/yyyy | en-US · MM/dd/yyyy | Verdict | Month (US) | Why this row behaves this way |
|---|---|---|---|---|---|---|
| 1 | 01/02/2026 | 2026-02-01 | 2026-01-02 | Wrong month | January | Day 01 becomes month 01; the swap is invisible because both numbers are valid months. |
| 2 | 02/02/2026 | 2026-02-02 | 2026-02-02 | Right by luck | February | Day and month are the same number, so the swap changes nothing — luck, not correctness. |
| 3 | 03/02/2026 | 2026-02-03 | 2026-03-02 | Wrong month | March | A valid month either way; nothing errors, nothing warns. |
| 4 | 04/02/2026 | 2026-02-04 | 2026-04-02 | Wrong month | April | As row 3. |
| 5 | 05/02/2026 | 2026-02-05 | 2026-05-02 | Wrong month | May | As row 3. |
| 6 | 06/02/2026 | 2026-02-06 | 2026-06-02 | Wrong month | June | As row 3. |
| 7 | 07/02/2026 | 2026-02-07 | 2026-07-02 | Wrong month | July | As row 3. |
| 8 | 09/02/2026 | 2026-02-09 | 2026-09-02 | Wrong month | September | Note the gap: there is no 08/02 row, so August never appears under either reading. |
| 9 | 10/02/2026 | 2026-02-10 | 2026-10-02 | Wrong month | October | As row 3. |
| 10 | 11/02/2026 | 2026-02-11 | 2026-11-02 | Wrong month | November | As row 3. |
| 11 | 12/02/2026 | 2026-02-12 | 2026-12-02 | Wrong month | December | The last swappable day: 12 is the largest number that is also a month. |
| 12 | 13/02/2026 | 2026-02-13 | Error | Error | Error | There is no month 13, so this is the one row that fails loudly — and the only reason the bug is findable at all. |

### The three written answers (6 marks)

**1. Which file is exposed, and how many loaded rows are at risk?**
Only the **marine** file — and therefore **12 loaded rows**. Its dates are text in
`dd/MM/yyyy`, the one format where day and month can be swapped. The other two files are not
exposed, for two different reasons: the rail file carries **real date values** in Excel (a date
typed as a date has no string to misread — its meaning does not depend on regional settings),
and the road file writes dates in **ISO `yyyy-mm-dd`**, where the year comes first and the
month's position is unambiguous, so no locale can swap day and month.

**2. Both reconciliation totals are exact and every monthly report is still wrong — why can't
the totals catch it?**
Because a date is not a measure. Changing a row's month moves it between report buckets without
changing its weight by a gram or its cost by a rupee, so `SUM(WeightKG)` and
`SUM(FreightCostPKR)` pass to the rupee while February's shipping is smeared across eleven
months. Totals reconcile *amounts*; only a check on the dates themselves — check 4, the month
set and the latest day — can catch a defect that lives entirely in a dimension. "Check the month
list" and "check the range" are not the same check, and neither is a total.

**3. The single click that prevents it, and the check that catches it if you forget.**
Prevent it with **Transform → Data Type → Using Locale → Date → English (United Kingdom)**
— never plain Change Type on a date column, not once, not on the road file that happens to be
ISO, because the habit is the lesson. In pandas the equivalent is stating the format explicitly:
`pd.to_datetime(..., format="%d/%m/%Y")` — `dayfirst=True` is a hint pandas may override, not a
statement. If you forget, **check 4** catches it: the distinct months of `ShipDate` must be
exactly {February} and the latest day must be 13 — under the US misreading the month set has
eleven members, and that is a failed check, not a mystery.